# 02 | Huấn luyện mô hình V6C: Stacking Extra Trees + Random Forest + XGBoost, đa domain

**Mục đích.** Tái tạo `models/multidomain/healthsense_af_v6c_stacking.pkl`, model nghiên cứu hiện hành (V6 Research Freeze) được báo cáo trong bài báo. Bản đọc được của `notebooks/v6/scripts/train_v6c_stacking.py`.

| | |
|---|---|
| Đầu vào | `data/features/v7/healthsense_af_v6c_pacaware_train.csv`: 137 476 cửa sổ, 248 bệnh nhân, 4 domain (MIMIC, DeepBeat, PulseWatch, Liu), nhãn `label_v7`, nhóm `subject_id` |
| Đầu ra | `models/multidomain/healthsense_af_v6c_stacking.pkl` (khoảng 1,4 GB) và `healthsense_af_v6c_stacking.json` |
| Thời gian | 6 lần fit mỗi base learner (5 fold + 1 lần cuối): 20 đến 40 phút trên CPU 8 nhân |
| Không nằm trong notebook | Platt scaling và Global rule V2: dò trên DEV rồi khoá, xem `notebooks/v6/protocol` |

Cách hoạt động: ba base learner cùng học 14 đặc trưng; xác suất **out-of-fold** của chúng (sinh bằng GroupKFold theo bệnh nhân) là đầu vào để meta-learner hồi quy logistic học cách gộp. Cuối cùng ba base learner được huấn luyện lại trên toàn bộ dữ liệu và đóng gói cùng meta-learner.

In [1]:
# Tìm gốc repo (chạy được từ thư mục notebook hoặc từ gốc) và nạp thư viện src/healthsense_ml
import os, sys, json, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "healthsense_ml").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Không tìm thấy gốc repo (thư mục chứa src/healthsense_ml)")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Repo root:", ROOT)

import numpy as np
import pandas as pd
import joblib
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

Repo root: D:\Projects\HealthSense\HealthSense-MachineLearning-Lab


In [2]:
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

DATA = ROOT / "data" / "features" / "v7" / "healthsense_af_v6c_pacaware_train.csv"
OUT_DIR = ROOT / "models" / "multidomain"
FEATURES = ["HR_mean", "Mean_NN", "SDNN", "RMSSD", "NN50", "pNN50", "CV",
            "HF", "Total_Power", "HF_norm", "SD1", "SD2", "SampEn", "PPG_AC"]

df = pd.read_csv(DATA, low_memory=False)
X = df[FEATURES]; y = df["label_v7"].astype(int); groups = df["subject_id"].astype(str)
print(f"{len(df)} cửa sổ | {groups.nunique()} bệnh nhân | AF {y.mean():.1%}")
df.groupby("domain").agg(cửa_sổ=("subject_id", "size"), bệnh_nhân=("subject_id", "nunique"),
                         tỉ_lệ_AF=("label_v7", "mean")).round(3)

137476 cửa sổ | 208 bệnh nhân | AF 23.3%


,cửa_sổ,bệnh_nhân,tỉ_lệ_AF
domain,,,
DEEPBEAT,19842,108,0.470
LIU,4363,31,0.509
MIMIC_PERFORM,4130,35,0.543
PULSEWATCH,109141,74,0.167


## 1. Ba base learner

Hai rừng dùng `class_weight="balanced"` để bù lệch lớp (AF chỉ chiếm khoảng 23%). Không giới hạn độ sâu nên mỗi cây mọc rất lớn, đó là lý do file model nặng; bảng đầu vào được dựng ở notebook 01.

In [3]:
def build_base_models():
    return {
        "ExtraTrees": ExtraTreesClassifier(n_estimators=500, class_weight="balanced", n_jobs=-1, random_state=42),
        "RandomForest": RandomForestClassifier(n_estimators=500, class_weight="balanced", n_jobs=-1, random_state=42),
        "XGBoost": XGBClassifier(n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.8,
                                 colsample_bytree=0.8, eval_metric="logloss", tree_method="hist", random_state=42),
    }
models = build_base_models()
list(models)

['ExtraTrees', 'RandomForest', 'XGBoost']

## 2. Xác suất out-of-fold theo bệnh nhân

Với mỗi fold, các base learner học trên 4/5 số bệnh nhân rồi dự đoán 1/5 còn lại. Ghép 5 lần lại, mỗi cửa sổ có ba xác suất do model **chưa từng thấy bệnh nhân đó** sinh ra. Đây là dữ liệu huấn luyện duy nhất của meta-learner, nhờ vậy meta-learner không bị thiên về các base learner đang học thuộc.

In [4]:
gkf = GroupKFold(n_splits=5)
oof = np.zeros((len(df), len(models)))
t_all = time.time()
for fold, (tr, va) in enumerate(gkf.split(X, y, groups), start=1):
    for j, (name, model) in enumerate(models.items()):
        t0 = time.time()
        model.fit(X.iloc[tr], y.iloc[tr])
        oof[va, j] = model.predict_proba(X.iloc[va])[:, 1]
        print(f"fold {fold}/5  {name:<12} {time.time()-t0:5.0f}s")
print(f"OOF xong trong {(time.time()-t_all)/60:.1f} phút")
oof_df = pd.DataFrame(oof, columns=list(models)); oof_df["y"] = y.values
oof_df.corr().round(3)

fold 1/5  ExtraTrees      13s


fold 1/5  RandomForest    38s


fold 1/5  XGBoost          7s


fold 2/5  ExtraTrees      12s


fold 2/5  RandomForest    32s


...

fold 4/5  XGBoost          6s


fold 5/5  ExtraTrees      12s


fold 5/5  RandomForest    36s


fold 5/5  XGBoost          7s
OOF xong trong 4.5 phút


,ExtraTrees,RandomForest,XGBoost,y
ExtraTrees,1.000,0.992,0.982,0.844
RandomForest,0.992,1.000,0.981,0.845
XGBoost,0.982,0.981,1.000,0.847
y,0.844,0.845,0.847,1.000


## 3. Meta-learner

Một hồi quy logistic trên ba cột xác suất, có chuẩn hoá. Hệ số sau khi fit cho biết meta-learner tin base learner nào hơn.

In [5]:
meta = Pipeline([("scale", StandardScaler()), ("lr", LogisticRegression(max_iter=1000))])
meta.fit(oof, y)
from sklearn.metrics import roc_auc_score
print("Hệ số meta-learner:", dict(zip(models, meta.named_steps["lr"].coef_[0].round(3))))
print("AUROC OOF của từng base learner:", {n: round(roc_auc_score(y, oof[:, j]), 4) for j, n in enumerate(models)})
print("AUROC OOF của stacking       :", round(roc_auc_score(y, meta.predict_proba(oof)[:, 1]), 4))

Hệ số meta-learner: {'ExtraTrees': np.float64(0.299), 'RandomForest': np.float64(1.735), 'XGBoost': np.float64(0.554)}
AUROC OOF của từng base learner: {'ExtraTrees': 0.9726, 'RandomForest': 0.9723, 'XGBoost': 0.9747}
AUROC OOF của stacking       : 0.9736


## 4. Huấn luyện lại base learner trên toàn bộ dữ liệu và đóng gói

Khi triển khai, ba base learner phải học từ **toàn bộ** 248 bệnh nhân (không còn chia fold), còn meta-learner giữ nguyên tham số đã học ở bước 3. Bundle lưu cả danh sách đặc trưng để AI-Service kiểm tra đúng thứ tự cột.

In [6]:
t0 = time.time()
for name, model in models.items():
    model.fit(X, y); print(f"final fit {name:<12} {time.time()-t0:5.0f}s")

OVERWRITE_FROZEN = False   # True: ghi đè đúng tên file đã đóng băng (bản dùng cho số liệu bài báo)
OUT_DIR.mkdir(parents=True, exist_ok=True)
stem = "healthsense_af_v6c_stacking" if OVERWRITE_FROZEN else "healthsense_af_v6c_stacking_retrained"
model_path = OUT_DIR / f"{stem}.pkl"
joblib.dump({"base_models": models, "meta": meta, "features": FEATURES}, model_path)
card = {
    "name": "HealthSense AF V6C Stacking", "version": "6c",
    "base_learners": {n: type(m).__name__ for n, m in models.items()}, "meta_learner": "StandardScaler + LogisticRegression",
    "oof_protocol": "GroupKFold(5) theo subject_id", "features": FEATURES,
    "training_windows": int(len(df)), "training_subjects": int(groups.nunique()),
    "domains": df.groupby("domain")["subject_id"].nunique().to_dict(),
    "oof_auroc_stack": round(float(roc_auc_score(y, meta.predict_proba(oof)[:, 1])), 4),
    "note": "Platt scaling và Global rule V2 được dò trên DEV và khoá riêng trong notebooks/v6/protocol.",
}
json.dump(card, open(OUT_DIR / f"{stem}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Đã lưu:", model_path, f"({model_path.stat().st_size/1e6:.0f} MB)")

final fit ExtraTrees      17s


final fit RandomForest    62s


final fit XGBoost         69s


Đã lưu: D:\Projects\HealthSense\HealthSense-MachineLearning-Lab\models\multidomain\healthsense_af_v6c_stacking_retrained.pkl (1877 MB)


## 5. Kiểm tra nhanh bundle vừa lưu

Nạp lại file, chạy dự đoán trên vài cửa sổ và so với meta-learner trong bộ nhớ.

In [7]:
bundle = joblib.load(model_path)
sample = X.iloc[:5]
base_probs = np.column_stack([bundle["base_models"][n].predict_proba(sample)[:, 1] for n in bundle["base_models"]])
pd.DataFrame(base_probs, columns=list(bundle["base_models"])).assign(
    meta_probability=bundle["meta"].predict_proba(base_probs)[:, 1], label=y.iloc[:5].values).round(4)

,ExtraTrees,RandomForest,XGBoost,meta_probability,label
0,0.0,0.326,0.2636,0.1238,0
1,0.0,0.050,0.0348,0.0268,0
2,0.0,0.114,0.3158,0.0535,0
3,0.0,0.054,0.0473,0.0278,0
4,0.0,0.034,0.0497,0.0254,0
